In [1]:
# =========================
# qcqp_generator_mixed.py  (UPDATED VERSION)
#
# ✅ Option A: keep ORIGINAL l/u (do NOT overwrite)
# ✅ If bounds_exist=False: inject ONE PD (ellipsoid) CONSTRAINT (k>=1) and
#    compute/store artificial bounds ONCE in l_art/u_art (so solver won't recompute)
# ✅ Increase linear objectives via p_linear_objective: with prob p, set A0 = 0
# ✅ [CHANGE 1] Add p_linear_constr: with prob p, set A_i = 0 for constraints i>=1
#    Target: g_linear_mass ~0.44 (was ~0.05), g1_Theta ~0.56 (was ~0.09)
# ✅ [CHANGE 5] Rebalanced p_convex_constr=0.65, p_concave_constr=0.15
#    Target: g0_Theta ~0.46 (was ~0.34), g_indef_Theta ~0.41 (was ~0.61)
# ✅ [CHANGE 2+7] Log-uniform eigenvalue range in diag_randI and eigvals_ints
#    exp(Uniform(log(0.01), log(500))) instead of integers(1, 11)
#    Target: h_avg_Lambda_max ~130 (was ~9), h_avg_LogCond ~7 (was ~2.6)
# ✅ [CHANGE 4] Post-generation scale factor in generate_matrix() and generate_psd_matrix()
#    Scale by exp(Uniform(log(0.001), log(0.3))) to pull rho away from 1.0
#    Target: h_avg_Rho ~0.095 (was ~0.92)
# ✅ [CHANGE 6] Wider log-uniform Laplacian edge weights in _pattern_laplacian_psd_from_pattern
#    Uniform(0.1, 50.0) log-uniformly instead of Uniform(0.5, 2.0)
#    Target: gD_PhiD ~0.48 (was ~0.12)
# ✅ [CHANGE 3] matrix_weights at call site: graph types↑, diag_ord1/rand1/eig_ones↓, spd↓
#    Target: h_avg_Xi ~0.055 (was ~0.33), comm_mean ~0.033 (was ~0.17)
#
# NOTE: Changes 3 are at the call site (bottom of file / usage script), not in the class.
# =========================

import numpy as np
import pandas as pd
from sklearn.datasets import make_spd_matrix
import networkx as nx
from networkx.algorithms import bipartite
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional, Tuple


# =============================================================================
# Graph / matrix helpers
# =============================================================================

def make_rng(seed: int):
    return np.random.default_rng(seed)


def adjacency_to_distance(A, rng, allow_zero_edge_weights: bool = False):
    if allow_zero_edge_weights:
        random_weights = rng.integers(-10, 11, size=A.shape)
    else:
        random_weights = rng.integers(1, 11, size=A.shape) * rng.choice([-1, 1], size=A.shape)

    D = np.where(A == 1, random_weights, 0)
    D = np.tril(D) + np.triu(D.T, 1)
    np.fill_diagonal(D, rng.integers(1, 11, size=D.shape[0]))
    return D, False


def generate_bipartite_distance_matrix(n, rng: np.random.Generator):
    k = rng.integers(2, n - 1)
    e = rng.integers(2, k * (n - k) + 1)
    seed_int = int(rng.integers(0, 2**31 - 1))
    B = bipartite.gnmk_random_graph(k, n - k, e, seed=seed_int)
    A = nx.to_numpy_array(B, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_tree_matrix(n: int, rng: np.random.Generator):
    seed_int = int(rng.integers(0, 2**31 - 1))
    T = (nx.random_labeled_tree(n, seed=seed_int) if hasattr(nx, "random_labeled_tree") else nx.random_tree(n, seed=seed_int))
    A = nx.to_numpy_array(T, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_planar_distance_matrix(n, rng: np.random.Generator):
    G = nx.complete_graph(n)
    edges = rng.permutation(list(G.edges)).tolist()
    is_planar = False
    while not is_planar and edges:
        edge = edges.pop()
        G.remove_edge(*edge)
        is_planar, _ = nx.check_planarity(G)
    if not nx.is_connected(G):
        remaining_edges = rng.permutation(list(nx.non_edges(G))).tolist()
        if remaining_edges:
            G.add_edge(*remaining_edges[0])
    A = nx.to_numpy_array(G, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_chordal_distance_matrix(n: int, rng: np.random.Generator):
    seed_int = int(rng.integers(0, 2**31 - 1))
    T = (nx.random_labeled_tree(n, seed=seed_int) if hasattr(nx, "random_labeled_tree") else nx.random_tree(n, seed=seed_int))
    for node in range(n):
        neighbors = list(nx.neighbors(T, node))
        if len(neighbors) >= 2:
            v, w = rng.choice(neighbors, size=2, replace=False)
            T.add_edge(v, w)
    A = nx.to_numpy_array(T, dtype=int)
    return adjacency_to_distance(A, rng)


def generate_symmetric_matrix(n: int, rng: np.random.Generator):
    A = rng.uniform(-5, 5, size=(n, n))
    D = 0.5 * (A + A.T)
    return D, False


def generate_spd_matrix(n: int, rng: np.random.Generator):
    seed_int = int(rng.integers(0, 2**31 - 1))
    D = make_spd_matrix(n, random_state=seed_int)
    return D, False


def _rand_Q(n, rng):
    Q, _ = np.linalg.qr(rng.uniform(0, 9, size=(n, n)))
    return Q


def diag_ordered_ones(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = np.ones(n)
    v[:n_neg] = -1
    return np.diag(v), False


def diag_random_ones(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = np.ones(n)
    if n_neg:
        v[rng.choice(n, n_neg, replace=False)] = -1
    return np.diag(v), False


# ── CHANGE 7a: diag_randI now uses log-uniform eigenvalue range ───────────────
# OLD: v = rng.integers(1, 11, size=n).astype(float)
# NEW: v = np.exp(rng.uniform(np.log(0.01), np.log(500), size=n))
# Reason: integers(1,11) gives h_avg_Lambda_max~9, target is ~130.
#         Log-uniform gives realistic spread with high IQR_LogCond (~11.7 target).
def diag_random_ints(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    # CHANGE 7a: log-uniform instead of uniform integers
    v = np.exp(rng.uniform(np.log(0.01), np.log(500), size=n))
    if n_neg:
        v[rng.choice(n, n_neg, replace=False)] *= -1
    return np.diag(v), False


def eigvals_ones(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    v = np.ones(n)
    v[:n_neg] = -1
    L = np.diag(v)
    Q = _rand_Q(n, rng)
    return Q @ L @ Q.T, False


# ── CHANGE 7b: eigvals_ints now uses log-uniform eigenvalue range ─────────────
# OLD: v = rng.integers(1, 11, size=n).astype(float)
# NEW: v = np.exp(rng.uniform(np.log(0.01), np.log(500), size=n))
# Same reason as diag_randI: need h_avg_Lambda_max ~130, h_IQR_LogCond ~11.7
def eigvals_ints(n, rng, n_neg=None):
    if n_neg is None:
        n_neg = int(rng.integers(0, n + 1))
    # CHANGE 7b: log-uniform instead of uniform integers
    v = np.exp(rng.uniform(np.log(0.01), np.log(500), size=n))
    if n_neg:
        v[rng.choice(n, n_neg, replace=False)] *= -1
    L = np.diag(v)
    Q = _rand_Q(n, rng)
    return Q @ L @ Q.T, False


# =============================================================================
# Artificial bounds (Cholesky-based)
# =============================================================================

def compute_artificial_bounds(A: np.ndarray, b: np.ndarray, c: float):
    """
    Bounds from ellipsoid:
        x^T A x + 2 b^T x + c <= 0
    assuming A is PD.
    """
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float).reshape(-1)
    n = A.shape[0]
    if A.shape != (n, n):
        raise ValueError("A must be square (n,n).")
    if b.size != n:
        raise ValueError("b has wrong dimension.")

    L = np.linalg.cholesky(A)

    def solve_A(rhs):
        rhs = np.asarray(rhs, dtype=float)
        y = np.linalg.solve(L, rhs)
        x = np.linalg.solve(L.T, y)
        return x

    Ainv_b = solve_A(b)
    b_Ainv_b = float(b @ Ainv_b)

    t = b_Ainv_b - float(c)
    if t < 0:
        raise ValueError("Invalid bounding: b^T A^-1 b - c must be non-negative.")

    sqrt_term = float(np.sqrt(max(t, 0.0)))

    Ainv = solve_A(np.eye(n))
    diag_Ainv = np.diag(Ainv)

    l = np.zeros(n, dtype=float)
    u = np.zeros(n, dtype=float)

    for i in range(n):
        di = float(diag_Ainv[i])
        if di < 0:
            raise ValueError("Numerical issue: negative diagonal entry in A^{-1}.")
        delta = sqrt_term * float(np.sqrt(max(di, 0.0)))
        center = -float(Ainv_b[i])
        l[i] = center - delta
        u[i] = center + delta

    return l, u


# =============================================================================
# CHANGE 6: Wider log-uniform Laplacian weights in PSD pattern constructor
# OLD: W = rng.uniform(0.5, 2.0, size=(n, n))
# NEW: W = np.exp(rng.uniform(np.log(0.1), np.log(50.0), size=(n, n)))
# Reason: narrow Uniform(0.5,2.0) gives weak diagonal dominance (gD_PhiD~0.12).
#         MINLPLib target is gD_PhiD~0.48. Log-uniform(0.1,50) creates high
#         variance in edge weights -> some constraints strongly diagonally dominant.
# =============================================================================

def _pattern_laplacian_psd_from_pattern(A_pattern: np.ndarray, rng: np.random.Generator, eps: float = 1e-8):
    """
    Build PSD matrix with same off-diagonal sparsity pattern as A_pattern using a weighted Laplacian.
    """
    A_pattern = np.asarray(A_pattern, dtype=float)
    n = A_pattern.shape[0]
    P = (A_pattern != 0).astype(float)
    np.fill_diagonal(P, 0.0)

    # CHANGE 6: log-uniform edge weights for realistic diagonal dominance variance
    W = np.exp(rng.uniform(np.log(0.1), np.log(50.0), size=(n, n)))  # was: rng.uniform(0.5, 2.0, ...)
    W = np.where(P > 0, W, 0.0)
    W = np.tril(W) + np.triu(W.T, 1)

    d = W.sum(axis=1)
    L = np.diag(d) - W  # PSD
    if eps > 0:
        L = L + eps * np.eye(n)
    return L


# =============================================================================
# Data structure
# =============================================================================

@dataclass
class QCQPInstance:
    instance_id: int
    bounds_exist: bool
    A_list: list
    b_list: list
    c_list: list
    l: np.ndarray
    u: np.ndarray
    results: dict = field(default_factory=dict)

    n: Optional[int] = None
    m: Optional[int] = None
    group_id: Optional[int] = None
    size_tag: Optional[str] = None

    l_art: Optional[np.ndarray] = None
    u_art: Optional[np.ndarray] = None


# =============================================================================
# Generator (single size)
# =============================================================================

class QCQPInstanceGenerator:
    def __init__(
        self,
        n,
        m,
        num_instances,
        seed,
        outcome_weights,
        matrix_weights=None,
        p_linear_objective: float = 0.86,
        # CHANGE 1: new parameter — probability a non-bilinear constraint i>=1 is linear (A=0)
        # OLD: no such parameter existed
        # NEW: p_linear_constr=0.45 targets g_linear_mass~0.44 and g1_Theta~0.56
        p_linear_constr: float = 0.45,
        # CHANGE 5: increased p_convex_constr 0.50->0.65, p_concave_constr 0.10->0.15
        # Reason: g0_Theta target 0.34->0.46, g_indef_Theta target 0.61->0.41
        # After ~45% go to linear, remaining non-linear split: 65% convex, 15% concave, 20% indef
        p_convex_constr: float = 0.65,   # was 0.50
        p_concave_constr: float = 0.15,  # was 0.10
        p_lowrank_quadratic: float = 0.65,
        lowrank_rank_range: tuple = (0.05, 0.45),
    ):
        self.n = int(n)
        self.m = int(m)
        self.master_seed = int(seed)
        self.num_instances = int(num_instances)
        self.outcome_weights = np.asarray(outcome_weights, dtype=float)
        self.matrix_weights = matrix_weights

        self.p_linear_objective = float(p_linear_objective)
        self.p_linear_constr    = float(p_linear_constr)    # CHANGE 1
        self.p_convex_constr    = float(p_convex_constr)    # CHANGE 5
        self.p_concave_constr   = float(p_concave_constr)   # CHANGE 5
        self.p_lowrank_quadratic = float(p_lowrank_quadratic)
        self.lowrank_rank_range = tuple(lowrank_rank_range)

    def _base_seed(self):
        return int(self.master_seed) + 1_000_003 * int(self.m) + 100_069 * int(self.n)

    def determine_outcome(self, instance_id):
        local_seed = self._base_seed() + int(instance_id) * 13 + 7
        rng = np.random.default_rng(local_seed)
        outcome = rng.choice(
            a=np.arange(1, len(self.outcome_weights) + 1),
            p=self.outcome_weights / np.sum(self.outcome_weights),
        )
        return int(outcome)

    # Synthetic' revision: turn OFF unobserved A-only post-scaling.
    # Keep this hook so all existing matrix-family functions remain unchanged.
    def _apply_scale(self, A: np.ndarray, rng: np.random.Generator) -> np.ndarray:
        return np.asarray(A, dtype=float)

    def generate_lowrank_matrix(self, family, rng):
        """Exactly low-rank symmetric PSD / NSD / indefinite matrix."""
        n = self.n
        lo, hi = self.lowrank_rank_range
        rank = int(np.clip(round(n * rng.uniform(lo, hi)), 1, n))
        if family == "indef":
            rank = max(2, rank)  # needs both signs
        rank = min(rank, n)
        # Sparse-support and diagonal cases retain genuinely sparse matrices.
        mode = rng.choice(["diagonal", "sparse_support", "shared_basis"], p=[.40, .40, .20])
        if mode == "diagonal":
            Q = np.eye(n)[:, rng.choice(n, size=rank, replace=False)]
        elif mode == "sparse_support":
            support_size = min(n, max(rank, int(np.ceil(1.6 * rank))))
            support = rng.choice(n, size=support_size, replace=False)
            Z = np.zeros((n, rank))
            Z[support] = rng.normal(size=(support_size, rank))
            Q, _ = np.linalg.qr(Z)
            Q = Q[:, :rank]
        else:
            Z = rng.normal(size=(n, rank))
            Q, _ = np.linalg.qr(Z)
            Q = Q[:, :rank]
        eig = np.exp(rng.uniform(np.log(.05), np.log(10.0), size=rank))
        if family == "nsd":
            eig = -eig
        elif family == "indef":
            signs = np.ones(rank)
            signs[:max(1, rank // 2)] = -1
            rng.shuffle(signs)
            eig *= signs
        elif family != "psd":
            raise ValueError(f"Unknown low-rank family: {family}")
        A = (Q * eig) @ Q.T
        return .5 * (A + A.T)

    def generate_matrix(self, matrix_type, bilinear, rng, n_neg=None):
        MATRIX_FUNCS = {
            "bipartite": generate_bipartite_distance_matrix,
            "tree": generate_tree_matrix,
            "planar": generate_planar_distance_matrix,
            "chordal": generate_chordal_distance_matrix,
            "symmetric": generate_symmetric_matrix,
            "diag_ord1": diag_ordered_ones,
            "diag_rand1": diag_random_ones,
            "diag_randI": diag_random_ints,
            "eig_ones": eigvals_ones,
            "eig_ints": eigvals_ints,
            "spd": generate_spd_matrix,
        }
        SIGN_TUNABLE = {"diag_ord1", "diag_rand1", "diag_randI", "eig_ones", "eig_ints"}

        if matrix_type in SIGN_TUNABLE:
            A, _ = MATRIX_FUNCS[matrix_type](self.n, rng, n_neg=n_neg)
        else:
            A, _ = MATRIX_FUNCS[matrix_type](self.n, rng)

        if bilinear:
            random_values = rng.uniform(-2, 3, size=(self.n, self.n))
            A = np.where(A != 0, random_values, 0.0)
            A = 0.5 * (A + A.T)
            np.fill_diagonal(A, 0.0)

        # CHANGE 4: apply log-uniform scale to pull rho away from 1.0
        A = self._apply_scale(A, rng)
        return A

    def generate_psd_matrix(self, matrix_type, rng):
        """
        PSD version of matrix_type.
        CHANGE 4 applied here too — scale after generation.
        """
        n = self.n
        SIGN_TUNABLE = {"diag_ord1", "diag_rand1", "diag_randI", "eig_ones", "eig_ints"}
        GRAPH_TYPES = {"bipartite", "tree", "planar", "chordal"}

        if matrix_type == "spd":
            A, _ = generate_spd_matrix(n, rng)
            # CHANGE 4
            return self._apply_scale(A, rng)

        if matrix_type in SIGN_TUNABLE:
            if matrix_type == "diag_ord1":
                A, _ = diag_ordered_ones(n, rng, n_neg=0)
            elif matrix_type == "diag_rand1":
                A, _ = diag_random_ones(n, rng, n_neg=0)
            elif matrix_type == "diag_randI":
                A, _ = diag_random_ints(n, rng, n_neg=0)
            elif matrix_type == "eig_ones":
                A, _ = eigvals_ones(n, rng, n_neg=0)
            elif matrix_type == "eig_ints":
                A, _ = eigvals_ints(n, rng, n_neg=0)
            # CHANGE 4
            return self._apply_scale(A, rng)

        if matrix_type in GRAPH_TYPES:
            if matrix_type == "bipartite":
                A_pat, _ = generate_bipartite_distance_matrix(n, rng)
            elif matrix_type == "tree":
                A_pat, _ = generate_tree_matrix(n, rng)
            elif matrix_type == "planar":
                A_pat, _ = generate_planar_distance_matrix(n, rng)
            else:
                A_pat, _ = generate_chordal_distance_matrix(n, rng)
            A = _pattern_laplacian_psd_from_pattern(A_pat, rng, eps=1e-8)
            # CHANGE 4
            return self._apply_scale(A, rng)

        # symmetric / fallback: PSD dense
        M = rng.uniform(-2.0, 2.0, size=(n, n))
        A = M.T @ M + 1e-8 * np.eye(n)
        A = 0.5 * (A + A.T)
        # CHANGE 4
        return self._apply_scale(A, rng)

    def generate_instance(self, instance_id):
        base = self._base_seed()
        rng = np.random.default_rng(base + int(instance_id) * 10 + 3)

        outcome = self.determine_outcome(instance_id)

        bounds_exist = outcome > 4
        matrix_combined = outcome in {1, 2, 5, 6}
        bilinear = outcome in {1, 3, 5, 7}

        sparsity_matrices = ["bipartite", "tree", "planar", "chordal", "symmetric", "eig_ones", "eig_ints"]
        non_eligible_matrices = ["spd", "diag_ord1", "diag_rand1", "diag_randI"]
        all_matrices = sparsity_matrices + non_eligible_matrices

        A_list, b_list, c_list = [], [], []
        matrix_types = sparsity_matrices if bilinear else all_matrices

        if matrix_combined:
            weights = np.array([self.matrix_weights[mt] for mt in matrix_types], dtype=float)
            prob = weights / weights.sum()
            chosen_matrix_types = rng.choice(matrix_types, size=self.m, replace=True, p=prob)
        else:
            weights = np.array([self.matrix_weights[mt] for mt in all_matrices], dtype=float)
            prob = weights / weights.sum()
            single_type = rng.choice(all_matrices, p=prob)
            chosen_matrix_types = [single_type] * self.m

        SIGN_TUNABLE = {"diag_ord1", "diag_rand1", "diag_randI", "eig_ones", "eig_ints"}
        num_neg_eigvals = []
        for mt in chosen_matrix_types:
            n_neg = int(rng.integers(0, self.n + 1)) if mt in SIGN_TUNABLE else None
            num_neg_eigvals.append(n_neg)

        pd_index = None
        l_art = None
        u_art = None
        if not bounds_exist:
            if self.m <= 1:
                raise ValueError("Artificial bounds require at least one constraint (m>=2 matrices)")
            pd_index = int(rng.integers(1, self.m))

        for i, matrix_type in enumerate(chosen_matrix_types):
            sub_rng = np.random.default_rng(base + int(instance_id) * 1000 + i * 97 + 11)
            n_neg = num_neg_eigvals[i]

            # (A) PD injection — unchanged, cannot be broken by linear objective
            if (not bounds_exist) and (i == pd_index):
                while True:
                    A_pd, _ = generate_spd_matrix(self.n, sub_rng)
                    b_pd = sub_rng.uniform(-5, 5, size=self.n)
                    c_pd = -sub_rng.uniform(0, 10)
                    try:
                        np.linalg.cholesky(A_pd)
                        l_new, u_new = compute_artificial_bounds(A_pd, b_pd, c_pd)
                        break
                    except Exception:
                        continue
                A_list.append(A_pd)
                b_list.append(b_pd)
                c_list.append(c_pd)
                l_art, u_art = l_new, u_new
                continue

            # (B) Objective: make linear with prob p_linear_objective — unchanged
            if i == 0 and (sub_rng.random() < self.p_linear_objective):
                A_list.append(np.zeros((self.n, self.n), dtype=float))
                b_list.append(sub_rng.uniform(-5, 5, size=self.n))
                c_list.append(-sub_rng.uniform(0, 10))
                continue

            # ── CHANGE 1: linear constraint injection ─────────────────────────
            # OLD: no linear constraint possibility existed
            # NEW: for non-bilinear constraints i>=1, with prob p_linear_constr set A=0
            # This drives g_linear_mass and g1_Theta toward MINLPLib targets.
            # Applied BEFORE convex/concave/indef decision so it takes full priority.
            if i >= 1 and (sub_rng.random() < self.p_linear_constr):
                A_list.append(np.zeros((self.n, self.n), dtype=float))
                b_list.append(sub_rng.uniform(-5, 5, size=self.n))
                c_list.append(-sub_rng.uniform(0, 10))
                continue

            # (C) Constraints: convex/concave/indef control (only if NOT bilinear)
            # CHANGE 5: p_convex_constr 0.50->0.65, p_concave_constr 0.10->0.15
            # These probabilities now apply to the REMAINING non-linear constraints
            # (after linear constraints have already been filtered out above).
            if i >= 1 and (sub_rng.random() < self.p_lowrank_quadratic):
                # New family; old families below remain unchanged.
                r = sub_rng.random()
                if bilinear:
                    family = "indef"
                elif r < self.p_convex_constr:
                    family = "psd"
                elif r < self.p_convex_constr + self.p_concave_constr:
                    family = "nsd"
                else:
                    family = "indef"
                A = self.generate_lowrank_matrix(family, sub_rng)
            elif i >= 1 and (not bilinear):
                r = sub_rng.random()
                if r < self.p_convex_constr:
                    A = self.generate_psd_matrix(matrix_type, sub_rng)
                elif r < self.p_convex_constr + self.p_concave_constr:
                    A = -self.generate_psd_matrix(matrix_type, sub_rng)
                else:
                    A = self.generate_matrix(matrix_type, bilinear, sub_rng, n_neg=n_neg)
            else:
                # Original bilinear family and objective behavior unchanged.
                A = self.generate_matrix(matrix_type, bilinear, sub_rng, n_neg=n_neg)

            A_list.append(A)
            b_list.append(sub_rng.uniform(-5, 5, size=self.n))
            c_list.append(-sub_rng.uniform(0, 10))

        # ORIGINAL bounds only
        if bounds_exist:
            l = np.zeros(self.n, dtype=float)
            u = np.ones(self.n, dtype=float)
        else:
            l = np.full(self.n, -np.inf, dtype=float)
            u = np.full(self.n,  np.inf, dtype=float)

        inst = QCQPInstance(int(instance_id), bool(bounds_exist), A_list, b_list, c_list, l, u)
        inst.n = self.n
        inst.m = self.m

        if not bounds_exist:
            inst.l_art = l_art
            inst.u_art = u_art

        return inst

    def save_instances(self, filename, start_id=0):
        instances = [self.generate_instance(i) for i in range(int(start_id), int(start_id) + self.num_instances)]
        pd.to_pickle(instances, filename)


# =============================================================================
# Generator (mixed sizes)
# =============================================================================

class QCQPSizeMixtureGenerator:
    """
    size_plan entries like:
        {"n": 5, "m_constr": 10, "count": 400}
    Optional overrides per entry:
        {"outcome_weights":..., "matrix_weights":...,
         "p_linear_objective":..., "p_linear_constr":...,
         "p_convex_constr":..., "p_concave_constr":...}
    """

    def __init__(
        self,
        size_plan: List[Dict[str, Any]],
        seed: int,
        default_outcome_weights: np.ndarray,
        default_matrix_weights: Dict[str, float],
        default_p_linear_objective: float = 0.86,
        # CHANGE 1: new default parameter propagated through mixture generator
        default_p_linear_constr: float = 0.45,
        # CHANGE 5: updated defaults
        default_p_convex_constr: float = 0.65,   # was 0.50
        default_p_concave_constr: float = 0.15,  # was 0.10
        default_p_lowrank_quadratic: float = 0.65,
        shuffle: bool = True,
        shuffle_seed_offset: int = 12345,
    ):
        self.master_seed = int(seed)
        self.size_plan = list(size_plan)
        self.default_outcome_weights = np.asarray(default_outcome_weights, dtype=float)
        self.default_matrix_weights = dict(default_matrix_weights)

        self.default_p_linear_objective = float(default_p_linear_objective)
        self.default_p_linear_constr    = float(default_p_linear_constr)    # CHANGE 1
        self.default_p_convex_constr    = float(default_p_convex_constr)    # CHANGE 5
        self.default_p_concave_constr   = float(default_p_concave_constr)   # CHANGE 5
        self.default_p_lowrank_quadratic = float(default_p_lowrank_quadratic)

        self._gens: Dict[int, QCQPInstanceGenerator] = {}
        jobs: List[Tuple[int, int]] = []

        for gid, spec in enumerate(self.size_plan):
            n = int(spec["n"])
            m_constr = int(spec["m_constr"])
            count = int(spec["count"])
            m_total = m_constr + 1

            ow  = np.asarray(spec.get("outcome_weights", self.default_outcome_weights), dtype=float)
            mw  = spec.get("matrix_weights", self.default_matrix_weights)
            plo = float(spec.get("p_linear_objective", self.default_p_linear_objective))
            plc = float(spec.get("p_linear_constr",    self.default_p_linear_constr))   # CHANGE 1
            pcv = float(spec.get("p_convex_constr",    self.default_p_convex_constr))   # CHANGE 5
            pcc = float(spec.get("p_concave_constr",   self.default_p_concave_constr))  # CHANGE 5
            plr = float(spec.get("p_lowrank_quadratic", self.default_p_lowrank_quadratic))
            rankrange = tuple(spec.get("lowrank_rank_range", (0.05, 0.45)))

            gen = QCQPInstanceGenerator(
                n=n,
                m=m_total,
                num_instances=count,
                seed=self.master_seed,
                outcome_weights=ow,
                matrix_weights=mw,
                p_linear_objective=plo,
                p_linear_constr=plc,   # CHANGE 1
                p_convex_constr=pcv,   # CHANGE 5
                p_concave_constr=pcc,  # CHANGE 5
                p_lowrank_quadratic=plr,
                lowrank_rank_range=rankrange,
            )
            self._gens[gid] = gen

            for local_id in range(count):
                jobs.append((gid, local_id))

        if shuffle:
            rng = np.random.default_rng(self.master_seed + int(shuffle_seed_offset))
            perm = rng.permutation(len(jobs))
            self.jobs = [jobs[i] for i in perm]
        else:
            self.jobs = jobs

        self.num_instances = len(self.jobs)

    def generate_instance(self, global_id: int) -> QCQPInstance:
        gid, local_id = self.jobs[int(global_id)]
        inst = self._gens[gid].generate_instance(local_id)

        inst.instance_id = int(global_id)
        inst.group_id = int(gid)
        inst.n = inst.l.size
        inst.m = len(inst.A_list)
        inst.size_tag = f"n{inst.n}_m{inst.m}"
        return inst

    def save_instances(self, filename: str, start_id: int = 0):
        start_id = int(start_id)
        end_id = min(self.num_instances, start_id + self.num_instances)
        instances = [self.generate_instance(i) for i in range(start_id, end_id)]
        pd.to_pickle(instances, filename)




In [2]:
# =============================================================================
# CELL 1 — BUILD + SAVE MIXED-SIZE PKL
# =============================================================================
if __name__ == "__main__":
    import os, gc, psutil, pickle

    # ── CHANGE 3: updated matrix weights ─────────────────────────────────────
    # OLD weights: bipartite=1, tree=1, planar=1, chordal=1, symmetric=1, spd=10,
    #              diag_ord1=1, diag_rand1=1, diag_randI=1, eig_ones=1, eig_ints=1
    #
    # NEW weights rationale:
    #   - bipartite/tree/planar/chordal: 1->3.0  (sparse, fixes h_avg_Xi KS=0.76,
    #     gD_PhiD KS=0.73, comm_mean KS=0.73; graph types naturally sparse)
    #   - spd: 10->1.0  (was dominant; too dense & well-conditioned; KS hurt)
    #   - symmetric: 1->0.5  (dense, too well-conditioned)
    #   - diag_ord1: 1->0.1  (always eigenvalues=±1 -> rho always=1.0, KS=0.92)
    #   - diag_rand1: 1->0.1 (same reason as diag_ord1)
    #   - eig_ones: 1->0.1   (same reason — unit eigenvalues lock rho=1.0)
    #   - diag_randI: 1->1.5 (now log-uniform so realistic; keep moderate weight)
    #   - eig_ints: 1->1.5   (now log-uniform so realistic; keep moderate weight)
    custom_matrix_weights = {
        "bipartite":  3.0,   # was 1  — sparse, realistic diagonal dominance
        "tree":       3.0,   # was 1  — sparse
        "planar":     3.0,   # was 1  — sparse
        "chordal":    3.0,   # was 1  — sparse
        "symmetric":  0.5,   # was 1  — dense/well-conditioned, reduce
        "spd":        1.0,   # was 10 — was dominant and too well-conditioned
        "diag_ord1":  0.1,   # was 1  — always rho=1, nearly removed
        "diag_rand1": 0.1,   # was 1  — always rho=1, nearly removed
        "diag_randI": 1.5,   # was 1  — now log-uniform, keep moderate
        "eig_ones":   0.1,   # was 1  — always rho=1, nearly removed
        "eig_ints":   1.5,   # was 1  — now log-uniform, keep moderate
    }

    # Original eight outcome types stay intact; only mixture weights vary.
    # Outcome IDs: 1..4 no original bounds, 5..8 explicit bounds.
    # Within each block: odd IDs bilinear; 1/2 and 5/6 mixed matrices.
    def outcome_mix(p_explicit, p_bilinear, p_mixed=0.75):
        probabilities = []
        for has_bounds in (False, True):
            for mixed, bilinear in ((True, True), (True, False),
                                    (False, True), (False, False)):
                probabilities.append(
                    (p_explicit if has_bounds else 1-p_explicit)
                    * (p_mixed if mixed else 1-p_mixed)
                    * (p_bilinear if bilinear else 1-p_bilinear)
                )
        return np.asarray(probabilities, dtype=float)

    outcome_weights = outcome_mix(.85, .35)

    # Two broad structural regimes, with varying n, m, and linear proportions.
    # Deliberate diverse starting design, NOT calibrated to held-out data.
    size_plan = []
    dimensions = [(5,10), (10,20), (20,10), (20,50),
                  (40,10), (40,50), (80,10), (80,30)]
    for regime in ("MIN_like", "QPLIB_like"):
        for n, m in dimensions:
            for band in (0, 1, 2):
                if regime == "MIN_like":
                    p_lin = (0.25, 0.45, 0.65)[band]
                    obj_lin, bilinear, lowrank = 0.85, 0.32, 0.70
                    bounds = 0.85
                    rank_range = (0.05, 0.35)
                else:
                    p_lin = (0.65, 0.80, 0.95)[band]
                    obj_lin, bilinear, lowrank = 0.25, 0.42, 0.60
                    bounds = 0.85
                    rank_range = (0.08, 0.50)
                size_plan.append({
                    "n": n, "m_constr": m, "count": 167,
                    "p_linear_objective": obj_lin,
                    "p_linear_constr": p_lin,
                    "p_lowrank_quadratic": lowrank,
                    "lowrank_rank_range": rank_range,
                    "outcome_weights": outcome_mix(bounds, bilinear),
                })
    # 2 regimes x 8 dimensions x 3 bands x 167 = 8,016 instances.

    # Final candidate: 48 strata x 167 instances = 8,016 QCQPs.
    # Use a different seed from pilot (42), retaining the same distributions.
    gen = QCQPSizeMixtureGenerator(
        size_plan=size_plan,
        seed=43,
        default_outcome_weights=outcome_weights,
        default_matrix_weights=custom_matrix_weights,
        shuffle=True,
    )

    from pathlib import Path
    import pandas as pd
    from time import perf_counter

    output_dir = Path("paperexperiments") / "synthetic_prime_minimal_revision"
    output_dir.mkdir(parents=True, exist_ok=True)
    out_pkl = output_dir / "instances_mixed_8016_candidate.pkl"
    if out_pkl.exists():
        raise FileExistsError(f"Refusing to overwrite existing file: {out_pkl}")

    total = gen.num_instances
    print(f"Generating {total} QCQPs into ONE PKL: {out_pkl.resolve()}", flush=True)
    print("Note: all QCQPs are held in RAM until the PKL is saved.", flush=True)
    t0 = perf_counter()
    instances = []
    for i in range(total):
        instances.append(gen.generate_instance(i))
        if (i + 1) % 100 == 0 or i + 1 == total:
            print(f"Generated {i+1}/{total}", flush=True)

    tmp_pkl = out_pkl.with_suffix(".pkl.partial")
    if tmp_pkl.exists():
        raise FileExistsError(f"Incomplete temporary file already exists: {tmp_pkl}")
    try:
        pd.to_pickle(instances, tmp_pkl)
        tmp_pkl.replace(out_pkl)
    finally:
        if tmp_pkl.exists():
            tmp_pkl.unlink()

    print(f"DONE: {len(instances)} instances saved to {out_pkl.resolve()} ({perf_counter()-t0:.1f}s)", flush=True)
    pkl_path = str(out_pkl)  # Existing root-solver cell can use this variable.


Generating 8016 QCQPs into ONE PKL: C:\Users\suuser\Desktop\minlplib\paperexperiments\synthetic_prime_minimal_revision\instances_mixed_8016_candidate.pkl
Note: all QCQPs are held in RAM until the PKL is saved.
Generated 100/8016
Generated 200/8016
Generated 300/8016
Generated 400/8016
Generated 500/8016
Generated 600/8016
Generated 700/8016
Generated 800/8016
Generated 900/8016
Generated 1000/8016
Generated 1100/8016
Generated 1200/8016
Generated 1300/8016
Generated 1400/8016
Generated 1500/8016
Generated 1600/8016
Generated 1700/8016
Generated 1800/8016
Generated 1900/8016
Generated 2000/8016
Generated 2100/8016
Generated 2200/8016
Generated 2300/8016
Generated 2400/8016
Generated 2500/8016
Generated 2600/8016
Generated 2700/8016
Generated 2800/8016
Generated 2900/8016
Generated 3000/8016
Generated 3100/8016
Generated 3200/8016
Generated 3300/8016
Generated 3400/8016
Generated 3500/8016
Generated 3600/8016
Generated 3700/8016
Generated 3800/8016
Generated 3900/8016
Generated 4000/8016

In [4]:

from qcqp_solversdg import QCQPSolver
import numpy as np
import pandas as pd
import pickle
import os
import gc
import psutil
from pathlib import Path

# ============================================================
# CELL 3 — LOAD FINAL PKL + SOLVE + WRITE CSV
# ============================================================

# New dataset — does NOT use or overwrite the pilot
BASE_DIR = Path("paperexperiments/synthetic_prime_minimal_revision")

pkl_path = BASE_DIR / "instances_mixed_8016_candidate.pkl"
out_csv = BASE_DIR / "instances_mixed_8016_root_results.csv"

if not pkl_path.exists():
    raise FileNotFoundError(f"Input PKL not found: {pkl_path}")

BASE_DIR.mkdir(parents=True, exist_ok=True)

print("Input PKL:", pkl_path)
print("Output CSV:", out_csv)


# ============================================================
# BOUNDS HANDLING — UNCHANGED
# ============================================================

def _has_finite_bounds(l, u):
    l = np.asarray(l, dtype=float).reshape(-1)
    u = np.asarray(u, dtype=float).reshape(-1)
    return (
        l.size > 0
        and u.size == l.size
        and np.all(np.isfinite(l))
        and np.all(np.isfinite(u))
    )


def _bounds_valid(l, u):
    l = np.asarray(l, dtype=float).reshape(-1)
    u = np.asarray(u, dtype=float).reshape(-1)

    if l.size == 0 or u.size == 0 or l.size != u.size:
        return False

    if not (np.all(np.isfinite(l)) and np.all(np.isfinite(u))):
        return False

    if not np.all(u > l):
        return False

    return True


def _working_bounds_for_solve(inst_d):
    l = np.asarray(inst_d.get("l", []), dtype=float).reshape(-1)
    u = np.asarray(inst_d.get("u", []), dtype=float).reshape(-1)
    bounds_exist = bool(inst_d.get("bounds_exist", False))

    if bounds_exist and _has_finite_bounds(l, u):
        return l, u

    l_art = inst_d.get("l_art", None)
    u_art = inst_d.get("u_art", None)

    if (
        l_art is not None
        and u_art is not None
        and _bounds_valid(l_art, u_art)
    ):
        return (
            np.asarray(l_art, dtype=float).reshape(-1),
            np.asarray(u_art, dtype=float).reshape(-1),
        )

    raise RuntimeError(
        "Missing or invalid artificial bounds for "
        "bounds_exist=False instance."
    )


# ============================================================
# SOLVE ALL INSTANCES, WITH RESUME SUPPORT
# ============================================================

def run_instances_from_pkl(
    pkl_path,
    out_csv,
    solve_lp=True,
    solve_sdp=True,
    solve_sdp_prime=True,
    solve_sdp_rlt=True,
    print_every=50,
    gc_every=200,
    solver_kwargs=None,
):
    if solver_kwargs is None:
        solver_kwargs = {}

    with open(pkl_path, "rb") as f:
        instances = pickle.load(f)

    print(f"Loaded {len(instances)} instances.")

    # --------------------------------------------------------
    # RESUME: retain existing results; skip completed IDs
    # --------------------------------------------------------

    completed_ids = set()
    out_csv = Path(out_csv)

    if out_csv.exists():
        if out_csv.stat().st_size == 0:
            raise RuntimeError(
                f"Existing CSV is empty: {out_csv}. "
                "Inspect it before restarting."
            )

        previous = pd.read_csv(out_csv)

        required = {
            "id", "lp_status", "sdp_status",
            "sdp_prime_status", "sdp_rlt_status"
        }
        if not required.issubset(previous.columns):
            raise ValueError(
                "Existing CSV has unexpected columns. "
                "Refusing to append incompatible results."
            )

        if previous["id"].isna().any():
            raise ValueError("Existing CSV has missing IDs.")

        if previous["id"].duplicated().any():
            raise ValueError("Existing CSV contains duplicate IDs.")

        completed_ids = set(previous["id"].astype(int))

        expected_ids = {
            int(
                (inst.__dict__ if hasattr(inst, "__dict__") else inst)
                .get("instance_id", i)
            )
            for i, inst in enumerate(instances)
        }

        if not completed_ids.issubset(expected_ids):
            raise ValueError(
                "Existing CSV contains IDs not found in this PKL. "
                "Refusing to mix datasets."
            )

        print(
            f"RESUMING: {len(completed_ids)} rows already saved. "
            f"{len(expected_ids - completed_ids)} remaining."
        )

    else:
        print("Starting new root-results CSV.")

    header_written = out_csv.exists()
    proc = psutil.Process(os.getpid())

    split = ""

    for idx, inst in enumerate(instances):
        inst_d = inst.__dict__ if hasattr(inst, "__dict__") else inst
        instance_id = int(inst_d.get("instance_id", idx))

        # Do not solve an instance whose row is already saved.
        if instance_id in completed_ids:
            continue

        solver = None

        try:
            A_list = inst_d["A_list"]
            b_list = inst_d["b_list"]
            c_list = inst_d["c_list"]

            l_work, u_work = _working_bounds_for_solve(inst_d)

            n = int(
                inst_d.get("n", None)
                or l_work.size
                or len(b_list[0])
            )

            m_total = int(
                inst_d.get("m", None) or len(A_list)
            )

            m_constr = m_total - 1

            solver = QCQPSolver(
                A_list,
                b_list,
                c_list,
                l_work,
                u_work,
                **solver_kwargs,
            )

            lp = (
                solver.solve("LP")
                if solve_lp
                else {
                    "objective": np.nan,
                    "time": np.nan,
                    "status": "skipped",
                }
            )

            sdp = (
                solver.solve("SDP")
                if solve_sdp
                else {
                    "objective": np.nan,
                    "time": np.nan,
                    "status": "skipped",
                }
            )

            sdp_p = (
                solver.solve("SDP_PRIME")
                if solve_sdp_prime
                else {
                    "objective": np.nan,
                    "time": np.nan,
                    "status": "skipped",
                }
            )

            sdp_rlt = (
                solver.solve("SDP_RLT")
                if solve_sdp_rlt
                else {
                    "objective": np.nan,
                    "time": np.nan,
                    "status": "skipped",
                }
            )

            if print_every and (idx % print_every == 0):
                print(
                    f"idx={idx} id={instance_id} "
                    f"| n={n} m_constr={m_constr} "
                    f"| LP={lp.get('objective')} "
                    f"| SDP={sdp.get('objective')} "
                    f"| SDP'={sdp_p.get('objective')} "
                    f"| SDP_RLT={sdp_rlt.get('objective')}",
                    flush=True,
                )

            row = {
                "id": instance_id,
                "split": split,
                "seed": inst_d.get("seed", None),
                "id_global": inst_d.get(
                    "id_global",
                    f"{instance_id}_n{n}_mc{m_constr}",
                ),
                "group_id": inst_d.get("group_id", None),
                "size_tag": inst_d.get("size_tag", None),
                "bounds_exist": inst_d.get("bounds_exist", None),
                "n": n,
                "m_total": m_total,
                "m_constr": m_constr,
                "lp_obj": lp.get("objective"),
                "lp_t": lp.get("time"),
                "lp_status": lp.get("status", "ok"),
                "sdp_obj": sdp.get("objective"),
                "sdp_t": sdp.get("time"),
                "sdp_status": sdp.get("status", "ok"),
                "sdp_prime_obj": sdp_p.get("objective"),
                "sdp_prime_t": sdp_p.get("time"),
                "sdp_prime_status": sdp_p.get("status", "ok"),
                "sdp_rlt_obj": sdp_rlt.get("objective"),
                "sdp_rlt_t": sdp_rlt.get("time"),
                "sdp_rlt_status": sdp_rlt.get("status", "ok"),
            }

        except Exception as e:
            print(f"⚠️ idx={idx} id={instance_id} failed: {e}", flush=True)

            row = {
                "id": instance_id,
                "split": split,
                "seed": inst_d.get("seed", None),
                "id_global": inst_d.get(
                    "id_global", f"err_{idx}"
                ),
                "group_id": inst_d.get("group_id", None),
                "size_tag": inst_d.get("size_tag", None),
                "bounds_exist": inst_d.get("bounds_exist", None),
                "n": inst_d.get("n", None),
                "m_total": inst_d.get("m", None),
                "m_constr": (
                    len(inst_d["A_list"]) - 1
                    if "A_list" in inst_d
                    else None
                ),
                "lp_obj": np.nan,
                "lp_t": np.nan,
                "lp_status": f"error:{e}",
                "sdp_obj": np.nan,
                "sdp_t": np.nan,
                "sdp_status": f"error:{e}",
                "sdp_prime_obj": np.nan,
                "sdp_prime_t": np.nan,
                "sdp_prime_status": f"error:{e}",
                "sdp_rlt_obj": np.nan,
                "sdp_rlt_t": np.nan,
                "sdp_rlt_status": f"error:{e}",
            }

        # Save each completed attempt immediately.
        pd.DataFrame([row]).to_csv(
            out_csv,
            mode="a",
            index=False,
            header=not header_written,
        )

        header_written = True
        completed_ids.add(instance_id)

        try:
            del solver
        except Exception:
            pass

        if gc_every and ((idx + 1) % gc_every == 0):
            gc.collect()

        if (idx + 1) % 500 == 0:
            print(
                "MEM_RSS_MB=",
                proc.memory_info().rss / 1e6,
                flush=True,
            )

    print(
        f"✅ Finished. {len(completed_ids)} rows saved to: {out_csv}"
    )


# ============================================================
# RUN FINAL CANDIDATE
# ============================================================

run_instances_from_pkl(
    pkl_path=pkl_path,
    out_csv=out_csv,
    print_every=50,
    solver_kwargs={
        "use_blocks": True,
        "block_tol": 1e-8,
        "verbose_blocks": False,
        "prune_matrices": False,
        "min_block_edges": 0,
    },
    solve_lp=True,
    solve_sdp=True,
    solve_sdp_prime=True,
    solve_sdp_rlt=True,
)


Input PKL: paperexperiments\synthetic_prime_minimal_revision\instances_mixed_8016_candidate.pkl
Output CSV: paperexperiments\synthetic_prime_minimal_revision\instances_mixed_8016_root_results.csv
Loaded 8016 instances.
Starting new root-results CSV.
Set parameter Username
Academic license - for non-commercial use only - expires 2027-09-30
idx=0 id=0 | n=80 m_constr=30 | LP=-109.42546652822453 | SDP=-inf | SDP'=-102.65429961076163 | SDP_RLT=nan
idx=50 id=50 | n=10 m_constr=20 | LP=-16.728951031616425 | SDP=-12.471798852371938 | SDP'=-12.471798744301292 | SDP_RLT=-12.471798683161607
idx=100 id=100 | n=5 m_constr=10 | LP=-12.276588620536636 | SDP=-12.266351119054105 | SDP'=-12.266351114394249 | SDP_RLT=-12.26635110695762
idx=150 id=150 | n=20 m_constr=10 | LP=-41.9411066545451 | SDP=-8.002969423740769 | SDP'=-8.002968550565825 | SDP_RLT=-8.002969004106632
idx=200 id=200 | n=5 m_constr=10 | LP=-6.049768946924306 | SDP=-6.049768951625208 | SDP'=-6.04976894969546 | SDP_RLT=-6.049768945658978

In [ ]:
"""import pandas as pd

# Load the saved instances
instances = pd.read_pickle("instances_5_100tr.pkl")

# Choose the instance you want to inspect
index = 0  # Change this to any index you want
inst = instances[index]

print(f"📌 Instance ID: {inst.instance_id}")
print(f"Bounds Exist?: {inst.bounds_exist}\n")"""

In [ ]:
import pandas as pd
df = pd.read_csv("instancesobj_20_2_10.csv")
print(df["split"].value_counts())
print(df["split"].unique()[:30])